# Docstring generator: QLoRA on Qwen2.5-Coder-1.5B

**Kaggle settings (right sidebar):** Accelerator → **GPU T4 x2** (or P100) · Internet → **On** · Persistence → Files (optional).
Optional: add a Kaggle Secret named `HF_TOKEN` (Add-ons → Secrets) to push models to Hugging Face or use gated datasets.

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
# one GPU is enough; avoids multi-GPU DataParallel surprises
!nvidia-smi

In [ ]:
!git clone -q https://github.com/Sidvortex/codedoc-lora.git || (cd codedoc-lora && git pull -q)
%cd codedoc-lora
!pip install -q -U -r requirements.txt

In [ ]:
# optional: log in to Hugging Face with a Kaggle secret
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret('HF_TOKEN'))
except Exception as e:
    print('No HF_TOKEN secret, continuing anonymously:', type(e).__name__)

## 1. Quick smoke run (≈5 min) to check everything works

In [ ]:
!python train.py --set data.max_train=200 data.max_eval=20 train.output_dir=outputs/smoke train.eval_steps=20 train.save_steps=1000

## 2. Full training (≈1.5–2 h on a T4)

In [ ]:
!python train.py --config configs/qwen_coder_1p5b.yaml

## 3. Base vs fine-tuned evaluation

In [ ]:
!python eval_model.py --config configs/qwen_coder_1p5b.yaml
import json; print(json.dumps(json.load(open('outputs/codedoc-qwen1.5b/results.json')), indent=2))

In [ ]:
from IPython.display import Markdown
Markdown(open('outputs/codedoc-qwen1.5b/samples.md').read()[:6000])

## 4. Try it on a file

In [ ]:
!python infer.py src/data.py --base | head -40   # base model
!python infer.py src/data.py | head -40          # fine-tuned

## 5. Save results (download from the Output tab)

In [ ]:
!cd outputs && zip -qr /kaggle/working/codedoc_outputs.zip codedoc-qwen1.5b -x '*/checkpoints/*'
!ls -lh /kaggle/working/*.zip